# RNN 人形机器人轨迹预测 · 学生填空版

EMEN 7029 / 7104 · PyTorch · PHUMA / Unitree G1

**任务：输入 20 个历史 XY 位置，预测未来 10 个位置（1 秒）。**
采用开源 PHUMA 的 G1 行走子集，手写 Elman RNN 和门控 RNN，对比匀速基线。
这是机器人**参考动作**的二维根节点预测，不是实机执行日志或全身控制。

上传本 notebook 到 [Google Colab](https://colab.research.google.com/)，使用 Python 3 CPU runtime。
数据和公共代码均已内置，单独一个 notebook 即可运行；无需账号下载数据。
本地建议 Python 3.12 + Jupyter。运行会在当前目录创建 `humanoid_rnn_workspace/`。

完成下列 13 个 TODO，然后从上到下运行。无需改动公共代码。留空处会主动报 NotImplementedError。

参考附件采用“任务说明 → TODO → 绘图验证 → 泛化分析”的形式，这里延续该教学组织。

## 0. 环境

默认在 CPU 上训练小模型。库已安装时此单元不会联网；缺失时自动安装。
完整实验每个模型训练 100 epochs，时间随机器而异。

In [ ]:
import importlib.util
import os
from pathlib import Path
import subprocess
import sys

requirements = {"numpy": "numpy>=1.26,<3", "torch": "torch>=2.2,<3",
                "matplotlib": "matplotlib>=3.8,<4"}
missing = [spec for name, spec in requirements.items()
           if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
ROOT = Path.cwd() / "humanoid_rnn_workspace"
ROOT.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".mplconfig"))
os.environ.setdefault("XDG_CACHE_HOME", str(ROOT / ".cache"))
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from IPython.display import Image, Markdown, display
torch.set_default_dtype(torch.float32)
print("Python:", sys.version.split()[0], "PyTorch:", torch.__version__, "NumPy:", np.__version__)
print("Workspace:", ROOT)

## 1. 数据选择与任务边界

核验日期：2026-10-30。本实验未使用 NGSIM，也未使用人工生成的 Bézier 曲线代替真实数据文件中的轨迹。

## 两个候选数据集

| 项目 | Kaggle Humanoid Robot Navigation | PHUMA（本实验采用） |
| --- | --- | --- |
| 原始链接 | https://www.kaggle.com/datasets/zara2099/humanoid-robot-navigation-dataset | https://arxiv.org/html/2510.26236v3 |
| 获取情况 | 实际下载并检查 CSV，1,000 行、13 列 | 官方 Hugging Face 公开发布，可固定版本获取 |
| 数据内容 | 导航状态、地形、能耗、稳定性、Bézier 控制点等 | G1/H1-2 的逐帧根节点位置、朝向、关节位置、帧率 |
| 连续位置预测适用性 | 缺少逐帧实际位置、采样间隔和可靠的 episode 边界，不适合直接构造本任务 | `root_trans` 和 `fps` 可直接构造历史/未来窗口 |
| 适合的简单任务 | 导航结果分类或分数回归；需另行核验数据来源 | G1 行走动作的根节点二维短期预测 |
| 发布方许可声明 | CC0 | Hugging Face 数据卡：Apache-2.0；同时保留上游来源限制 |

Kaggle CSV 的实际字段：`time_step, robot_id, terrain_type, obstacle_count, obstacle_distance, bezier_control_points, planned_trajectory_length, mpc_adjustment, gait_stability_index, center_of_mass_offset, energy_consumption, navigation_status, navigation_score`。
字段核验记录和两行样本见 `data/sources/kaggle_schema_verified.json`。`time_step` 与 `robot_id` 的存在本身不能证明各行构成等间隔的连续运动。Bézier 控制点描述规划路径，不能当成已执行轨迹的观测点。

## PHUMA 来源

- 论文：Lee et al., *PHUMA: Physically Reliable Humanoid Locomotion Dataset*, arXiv:2510.26236v3。
- 官方代码：https://github.com/DAVIAN-Robotics/PHUMA
- 项目页：https://davian-robotics.github.io/PHUMA/
- 官方数据：https://huggingface.co/datasets/DAVIAN-Robotics/PHUMA
- 本次固定 revision：`48b0f5982969104ab041461e594a8b8c36db8ba5`。
- 原始结构：`root_trans (T,3)`、`root_ori (T,4)`、`dof_pos (T,D)`、`fps`。

PHUMA 将人类动作数据经过物理约束处理并重定向到人形机器人。公开文件中的这些轨迹是**机器人参考动作**，不能描述为实机传感器测得的执行日志。论文的主要任务是动作模仿与控制；这里另行设计的是有监督轨迹预测教学实验，不是复现论文主实验。

## 可复现的小型子集

选取 `data/g1/idea400/` 中文件名包含 `walking` 且匹配 `number_|hand_pockets|arm_swing|clapping|salute|waving|pointing` 的动作。内容为行走及简单上肢动作，排除其他机器人、其他来源以及复杂运动类别。不是严格的“直线匀速行走”标签。

共 295 个片段、169 个按名称归并的源动作、29,856 个原始 30 Hz 帧。每隔 3 帧采样一次，得到 10 Hz 的 10,006 帧；保存 XYZ，模型只用水平 XY。NPZ 约 111 KiB，内置于两个 notebook，无需账号或下载完整约 3.4 GB 压缩包。`prepare_phuma.py` 可通过 HTTP Range 从固定官方版本重新提取同一子集。

筛选不依赖预测误差。每个原始文件的路径、SHA-256、CRC、帧率及长度均记录在 `data/manifest.json`。已逐一核对缓存的 295 个原始文件哈希，并验证 NPZ 坐标与原始 `root_trans[::3]` 完全一致。读取课堂 NPZ 使用 `allow_pickle=False`；重建工具仅允许原始 NPY 字典所需的 NumPy 类型。

以去掉 `_clip...` / `_chunk...` 后缀的源名称分组，先划分再滑窗；不同片段不拼接。默认窗口需要至少 30 帧，故最终有 157 个源动作可用：训练 109、验证 23、测试 25；对应窗口数 794、172、191。其余 12 个源动作不满足长度要求。

## 结果解释的限制

1. 按源名称隔离可避免同一源动作的分块进入不同集合；缺少可靠的演员 ID，不能声称演员身份完全独立，也不能排除不同名称动作间的相关性。
2. 相邻滑窗有重叠。报告同时提供逐窗口平均和逐源动作等权平均，不能把所有窗口当作独立样本来计算置信区间。
3. 我们没有新增平滑或插值；PHUMA 上游本身包含动作处理与物理优化。此结果反映已处理参考轨迹的可预测性，不代表带噪实机在线传感器表现。
4. XY 预测不包含全身关节、朝向、障碍物、接触和控制输入，不能直接用于机器人控制或保证动力学可行性。未来意图的多解性也没有被确定性 RNN 建模。
5. 当前是一个固定划分、固定种子的课堂结果。跨场景、跨来源、跨机器人、多种子与更长预测时域需要独立评估。

## 许可与引用

保留 `data/sources/PHUMA_DATA_CARD.md`、`PHUMA_LICENSE.txt` 和 `PHUMA_REPOSITORY_README.md`。官方数据卡明确说明公开包不包含受限制的 LAFAN1 与 LocoMuJoCo 衍生动作；本子集仅取公开包中的 IDEA400 来源。Apache-2.0 是发布方对该数据仓库的声明，不意味着可以忽略原始人体动作来源的权利或把论文的 CC BY 4.0 当成数据许可。进一步公开分发或商业使用时应核对上游 Motion-X / IDEA400 条款。

```bibtex
@article{lee2025phuma,
  title={PHUMA: Physically Reliable Humanoid Locomotion Dataset},
  author={Lee, Kyungmin and Kim, Sibeen and Lee, Youngdo and Park, Minho
          and Kim, Hyunseung and Hwang, Dongyoon and Kim, Donghu
          and Lee, Hojoon and Choo, Jaegul},
  journal={arXiv preprint arXiv:2510.26236},
  year={2025}
}
```


下面的折叠单元仅解包公开数据、来源清单及许可文本，不包含模型答案。无需编辑。

In [ ]:
# @title Restore the bundled public motion subset and provenance
import base64
import hashlib
import io
import json
import zipfile

DATA_B64 = "..."
archive_bytes = base64.b64decode(DATA_B64)
assert hashlib.sha256(archive_bytes).hexdigest() == "fa203945c2abbe05f57e5cae7c096bcfb2a1a07e97586bc455445f7a5d6e160a"
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    for name in archive.namelist():
        destination = (ROOT / name).resolve()
        if not destination.is_relative_to(ROOT.resolve()):
            raise ValueError("Unsafe archive path")
        content = archive.read(name)
        if destination.exists() and destination.read_bytes() != content:
            raise FileExistsError(f"Existing file differs: {destination}; use a new workspace")
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(content)
DATA = ROOT / "data" / "phuma_g1_walking.npz"
manifest = json.loads((DATA.parent / "manifest.json").read_text())
assert hashlib.sha256(DATA.read_bytes()).hexdigest() == manifest["sha256_npz"]
print("Data:", manifest["dataset"], manifest["robot"])
print("Sequences / source groups / frames:", manifest["num_sequences"],
      manifest["num_source_groups"], manifest["frames_10hz"])
print("Revision:", manifest["revision"])
del DATA_B64, archive_bytes

## 2. 模型设计与评分

目标：在 PHUMA 的 Unitree G1 行走子集上，手写 RNN，使用历史平面位置预测未来 1 秒的根节点轨迹。完成学生 notebook 的 13 个 TODO 后，它应与完整教师版具有相同功能。公共数据读取、分组划分、标准化、验证集选择与绘图均已提供。

本作业使用 `f(h,x)`、`g(h)`、`initial_recurrent_state(batch_size)` 和 `scan` 接口。为学习递归与梯度传播，不调用 `nn.RNN/GRU/LSTM` 及其 Cell，也不覆盖模型的 `forward`；可以使用 `nn.Linear`、张量运算和优化器。

## 1. 数据与时间定义

数据来自 [PHUMA 官方公开发布](https://huggingface.co/datasets/DAVIAN-Robotics/PHUMA)，为重定向的机器人参考动作，而非实机执行日志。采用 G1 的简单行走/伴随上肢动作，原始 `root_trans` 的水平 XY 单位为米，30 Hz 每三帧取一帧成为 10 Hz。不要修改固定数据或通过预测误差筛选测试样本。

- `B`：batch size；`O=20`：历史点数；`H=10`：未来点数。
- 历史时刻为 `-1.9,-1.8,...,0` 秒，覆盖 1.9 秒间隔，通常称约 2 秒历史。
- 未来时刻为 `0.1,0.2,...,1.0` 秒；`dt=0.1`。
- 特征维数 `D_x=4`，隐层 `D_h=32`，输出维数 `D_y=2H=20`。
- 同一源动作的 clips/chunks 留在同一集合；先划分源动作再滑窗，窗口不能跨片段。
- 仅训练集估计均值、标准差；输入、速度、基线和推理均不读取未来真值。

## 2. 任务与评分

| 任务 | 分值 | TODO 标识 |
| --- | ---: | --- |
| 历史位置和速度特征 | 10 | `features` |
| Elman RNN 参数、更新、输出、初始状态 | 20 | `elman_init / elman_f / elman_g / elman_h0` |
| 保留梯度的时间扫描 | 15 | `scan` |
| 未来多步残差预测 | 10 | `predict` |
| 一个 epoch 的训练循环 | 15 | `train` |
| 米制 ADE / FDE | 10 | `metrics` |
| 自定义门控 RNN | 15 | `custom_init / custom_f / custom_g / custom_h0` |
| 实验结果与解释 | 5 | 报告 |

## 3. 公共实验代码（已提供）

流程：按源动作划分 → 各片段内滑窗 → 训练集标准化 → 历史匀速基线 →
RNN 残差预测 → 验证 ADE 选模型 → 最后测试。

以下函数负责实验基础设施；学生只需完成后面的模型与训练函数。
公共训练组织函数会调用你实现的 `train_one_epoch`，不会绕过它。

In [ ]:
"""Provided infrastructure shared by the instructor and student versions."""

import argparse
import hashlib
import json
import os
from pathlib import Path
import random
import time

import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path.cwd() / "humanoid_rnn_workspace"
DATA = ROOT / "data" / "phuma_g1_walking.npz"
DT = 0.1
OUTPUT_SCALE = 1.0
DATA_URL = "https://huggingface.co/datasets/DAVIAN-Robotics/PHUMA"


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.use_deterministic_algorithms(True)


def read_tracks(path):
    """Group separate motion chunks by their original source recording."""
    if not path.exists():
        raise FileNotFoundError("Use bundled data or run python prepare_phuma.py first.")
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    manifest_path = path.with_name("manifest.json")
    manifest = json.loads(manifest_path.read_text())
    if digest != manifest["sha256_npz"]:
        raise ValueError("Dataset checksum differs from manifest.json.")
    with np.load(path, allow_pickle=False) as data:
        xyz, offsets, groups, names = (data[k] for k in ("root_xyz", "offsets", "groups", "names"))
        if float(data["fps"]) != 1 / DT:
            raise ValueError("Expected 10 Hz positions.")
        if (xyz.ndim != 2 or xyz.shape[1] != 3 or not np.isfinite(xyz).all()
                or offsets.shape != (len(groups) + 1,) or len(names) != len(groups)
                or offsets[0] != 0 or offsets[-1] != len(xyz)
                or not np.all(np.diff(offsets) > 0)):
            raise ValueError("Malformed motion arrays.")
        tracks = {}
        for i, group in enumerate(groups):
            # Never concatenate chunks: there may be temporal gaps between them.
            tracks.setdefault(str(group), []).append(xyz[offsets[i]:offsets[i+1], :2].copy())
    return tracks, {"sha256_npz": digest, "sequences": len(groups), "source_groups": len(tracks),
                    "revision": manifest["revision"], "raw_frames": sum(m["frames_raw"] for m in manifest["sequences"])}


def split_tracks(tracks, seed, minimum_length):
    ids = np.array([i for i in sorted(tracks)
                    if any(len(segment) >= minimum_length for segment in tracks[i])])
    if len(ids) < 15:
        raise ValueError("At least 15 sufficiently long source recordings are required.")
    np.random.default_rng(seed).shuffle(ids)
    n_train, n_val = int(0.7 * len(ids)), int(0.15 * len(ids))
    splits = {"train": ids[:n_train], "val": ids[n_train:n_train + n_val], "test": ids[n_train + n_val:]}
    assert len(set(np.concatenate(list(splits.values())))) == len(ids)
    return {name: sorted(map(str, values)) for name, values in splits.items()}


def make_windows(tracks, ids, obs_len, pred_len, stride):
    observed, future, source_ids = [], [], []
    for source in ids:
        for segment in tracks[source]:
            for start in range(0, len(segment) - obs_len - pred_len + 1, stride):
                observed.append(segment[start:start + obs_len])
                future.append(segment[start + obs_len:start + obs_len + pred_len])
                source_ids.append(source)
    if not observed:
        raise ValueError("Split contains no complete windows.")
    return torch.from_numpy(np.stack(observed)), torch.from_numpy(np.stack(future)), source_ids


def constant_velocity(observed, pred_len, dt=DT):
    """OLS velocity over the last 0.5 seconds; anchor prediction at last observation."""
    count = min(observed.shape[1], round(0.5 / dt) + 1)
    times = torch.arange(count, device=observed.device, dtype=observed.dtype) * dt
    times = times - times.mean()
    velocity = (observed[:, -count:] * times[None, :, None]).sum(1) / times.square().sum()
    horizon = torch.arange(1, pred_len + 1, device=observed.device,
                           dtype=observed.dtype) * dt
    return velocity[:, None] * horizon[None, :, None]


@torch.no_grad()
def evaluate(model, loader, predict, errors, device):
    model.eval()
    predictions, ade, fde = [], [], []
    for xs, baseline, target in loader:
        prediction = predict(model, xs.to(device), baseline.to(device), OUTPUT_SCALE).cpu()
        a, f = errors(prediction, target)
        predictions.append(prediction)
        ade.append(a)
        fde.append(f)
    return {"ADE_m": torch.cat(ade).mean().item(), "FDE_m": torch.cat(fde).mean().item()}, torch.cat(predictions)


def plot_results(out, histories, observed, target, predictions, baseline, source_ids):
    os.environ.setdefault("MPLCONFIGDIR", str(out / ".mplconfig"))
    os.environ.setdefault("XDG_CACHE_HOME", str(out / ".cache"))
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(7, 4))
    for name, history in histories.items():
        ax.plot(range(1, len(history) + 1), [x["val_ADE_m"] for x in history], label=name)
    ax.set(xlabel="Epoch", ylabel="Validation ADE (m)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out / "learning_curves.png", dpi=160)
    plt.close(fig)
    # Fixed indices chosen without looking at prediction errors.
    indices = np.linspace(0, len(target) - 1, 6, dtype=int)
    fig, axes = plt.subplots(2, 3, figsize=(13, 8))
    for ax, index in zip(axes.flat, indices):
        obs = observed[index] - observed[index, -1:]
        ax.plot(obs[:, 0], obs[:, 1], "o-", ms=2, color="0.5", label="History")
        for label, values, style in [("Truth", target, "k-"), ("CV", baseline, "--")]:
            points = torch.cat((torch.zeros(1, 2), values[index]))
            ax.plot(points[:, 0], points[:, 1], style, label=label)
        for name, values in predictions.items():
            points = torch.cat((torch.zeros(1, 2), values[index]))
            ax.plot(points[:, 0], points[:, 1], label=name)
        ax.set(title=f"Test window {index}", xlabel="Root x displacement (m)", ylabel="Root y displacement (m)")
        ax.set_aspect("equal", adjustable="datalim")
        ax.grid(alpha=0.2)
    axes.flat[0].legend(fontsize=8)
    fig.suptitle(f"PHUMA G1 held-out recordings: {target.shape[1] * DT:g}-second root forecast")
    fig.tight_layout()
    fig.savefig(out / "trajectories.png", dpi=160)
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(7, 4))
    for name, forecast in {"constant velocity": baseline, **predictions}.items():
        error = torch.linalg.vector_norm(forecast - target, dim=-1).mean(0)
        ax.plot(np.arange(1, len(error) + 1) * DT, error, label=name)
    ax.set(xlabel="Time after last observation (s)", ylabel="Mean displacement error (m)")
    ax.legend()
    ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(out / "error_by_horizon.png", dpi=160)
    plt.close(fig)


def run_experiment(model_classes, feature_fn, predict, train_one_epoch, errors, argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data", type=Path, default=DATA)
    parser.add_argument("--output", type=Path, default=ROOT / "outputs" / "my_run")
    parser.add_argument("--epochs", type=int, default=100)
    parser.add_argument("--batch-size", type=int, default=128)
    parser.add_argument("--hidden", type=int, default=32)
    parser.add_argument("--lr", type=float, default=0.001)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--obs-len", type=int, default=20)
    parser.add_argument("--pred-len", type=int, default=10)
    parser.add_argument("--stride", type=int, default=2)
    parser.add_argument("--model", choices=["both", "elman", "gated"], default="both")
    parser.add_argument("--device", choices=["cpu", "cuda"], default="cpu")
    parser.add_argument("--threads", type=int, default=2)
    args = parser.parse_args(argv)
    if min(args.epochs, args.batch_size, args.hidden, args.pred_len, args.stride, args.threads) < 1 or args.obs_len < 2 or args.lr <= 0:
        parser.error("Lengths/counts/lr must be positive, and obs-len must be >= 2.")
    if (args.output / "metrics.json").exists():
        parser.error("Output already contains an experiment; choose a new --output directory.")
    args.output.mkdir(parents=True, exist_ok=True)
    if args.device == "cuda":
        os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    torch.set_num_threads(args.threads)
    seed_everything(args.seed)
    tracks, provenance = read_tracks(args.data)
    splits = split_tracks(tracks, args.seed, args.obs_len + args.pred_len)
    windows = {s: make_windows(tracks, ids, args.obs_len, args.pred_len, args.stride)
               for s, ids in splits.items()}
    features = {s: feature_fn(w[0], DT) for s, w in windows.items()}
    mean = features["train"].mean(dim=(0, 1), keepdim=True)
    std = features["train"].std(dim=(0, 1), keepdim=True).clamp_min(1e-3)
    datasets = {}
    for s, (observed, future, _) in windows.items():
        datasets[s] = TensorDataset((features[s] - mean) / std,
                                    constant_velocity(observed, args.pred_len),
                                    future - observed[:, -1:])
    counts = {s: {"source_recordings": len(splits[s]), "windows": len(datasets[s])} for s in splits}
    print(json.dumps(counts, indent=2), flush=True)
    config = {k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items()}
    report = {"config": config, "dt_seconds": DT, "counts": counts,
              "data": {**provenance, "url": DATA_URL}, "split_source_ids": splits,
              "versions": {"torch": str(torch.__version__), "numpy": np.__version__},
              "test": {}, "models": {}}
    (args.output / "splits.json").write_text(json.dumps(splits, indent=2))
    val_loader = DataLoader(datasets["val"], batch_size=args.batch_size)
    histories, models = {}, {}
    val_cv = errors(datasets["val"].tensors[1], datasets["val"].tensors[2])
    report["validation_constant_velocity"] = {"ADE_m": val_cv[0].mean().item(), "FDE_m": val_cv[1].mean().item()}
    print("Validation CV:", report["validation_constant_velocity"], flush=True)
    for name, model_class in model_classes.items():
        if args.model not in ("both", name):
            continue
        seed_everything(args.seed)
        loader = DataLoader(datasets["train"], batch_size=args.batch_size, shuffle=True,
                            generator=torch.Generator().manual_seed(args.seed))
        model = model_class(4, args.hidden, args.pred_len * 2).to(args.device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
        best, history, started = float("inf"), [], time.perf_counter()
        for epoch in range(1, args.epochs + 1):
            loss = train_one_epoch(model, loader, optimizer, args.device, OUTPUT_SCALE)
            metric, _ = evaluate(model, val_loader, predict, errors, args.device)
            if not np.isfinite(loss) or not np.isfinite(metric["ADE_m"]):
                raise RuntimeError("Non-finite training/validation metric.")
            history.append({"epoch": epoch, "train_normalized_MSE": loss,
                            "val_ADE_m": metric["ADE_m"], "val_FDE_m": metric["FDE_m"]})
            if metric["ADE_m"] < best:
                best = metric["ADE_m"]
                checkpoint = {"model": name, "state_dict": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                              "mean": mean, "std": std, "config": config, "dt": DT,
                              "output_scale": OUTPUT_SCALE, "epoch": epoch,
                              "val_ADE_m": best, "data_sha256": provenance["sha256_npz"]}
                torch.save(checkpoint, args.output / f"{name}_best.pt")
            if epoch == 1 or epoch % 5 == 0 or epoch == args.epochs:
                print(f"{name:5} {epoch:3}/{args.epochs}: loss={loss:.5f}, "
                      f"val ADE={metric['ADE_m']:.3f} m, FDE={metric['FDE_m']:.3f} m", flush=True)
        model.load_state_dict(checkpoint["state_dict"])
        histories[name], models[name] = history, model
        report["models"][name] = {"best_epoch": checkpoint["epoch"], "val_ADE_m": best,
                                  "training_seconds": time.perf_counter() - started}
    # Test targets are used for final reporting only, after all model selection.
    test_loader = DataLoader(datasets["test"], batch_size=args.batch_size)
    _, baseline, target = datasets["test"].tensors
    for name, forecast in (("constant_velocity", baseline), ("last_position", torch.zeros_like(target))):
        ade, fde = errors(forecast, target)
        report["test"][name] = {"ADE_m": ade.mean().item(), "FDE_m": fde.mean().item()}
    predictions = {}
    for name, model in models.items():
        report["test"][name], predictions[name] = evaluate(model, test_loader, predict, errors, args.device)
    # Also report equally weighted source means; overlapping windows are correlated.
    report["test_macro_source"] = {}
    source_array = np.array(windows["test"][2])
    for name, forecast in {"constant_velocity": baseline, "last_position": torch.zeros_like(target), **predictions}.items():
        a, f = errors(forecast, target)
        report["test_macro_source"][name] = {
            "ADE_m": float(np.mean([a[source_array == group].mean().item() for group in splits["test"]])),
            "FDE_m": float(np.mean([f[source_array == group].mean().item() for group in splits["test"]])),
        }
    report["history"] = histories
    (args.output / "metrics.json").write_text(json.dumps(report, indent=2))
    np.savez_compressed(args.output / "test_predictions.npz", observed=windows["test"][0].numpy(),
                        target_relative=target.numpy(), baseline_relative=baseline.numpy(),
                        source_ids=np.array(windows["test"][2]),
                        **{k + "_relative": v.numpy() for k, v in predictions.items()})
    plot_results(args.output, histories, windows["test"][0], target, predictions, baseline, windows["test"][2])
    print("Final held-out test metrics (meters):", flush=True)
    print(json.dumps(report["test"], indent=2), flush=True)
    print(f"Artifacts: {args.output.resolve()}", flush=True)

In [ ]:
tracks, provenance = read_tracks(DATA)
splits = split_tracks(tracks, seed=42, minimum_length=30)
for name, ids in splits.items():
    observed, future, _ = make_windows(tracks, ids, 20, 10, 2)
    print(name, "source recordings:", len(ids), "windows:", len(observed),
          "history:", tuple(observed.shape), "future:", tuple(future.shape))
assert not (set(splits["train"]) & set(splits["test"]))
assert not (set(splits["val"]) & set(splits["test"]))
assert not (set(splits["train"]) & set(splits["val"]))

# Only a training trajectory is used for this preliminary data visualization.
import matplotlib.pyplot as plt
observed, future, _ = make_windows(tracks, splits["train"], 20, 10, 2)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(observed[0, :, 0], observed[0, :, 1], "o-", label="Observed: -1.9 to 0 s")
connected = torch.cat((observed[0, -1:], future[0]))
ax.plot(connected[:, 0], connected[:, 1], "o-", label="Future: +0.1 to +1.0 s")
ax.set(xlabel="Root x (m)", ylabel="Root y (m)", title="PHUMA G1: one training window")
ax.axis("equal")
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
example_path = ROOT / "training_example.png"
fig.savefig(example_path, dpi=140)
plt.close(fig)
display(Image(filename=str(example_path)))

## A. 历史特征，10 分

`make_features(observed,dt)`：输入 `(B,O,2)`，输出 `(B,O,4)`。前两维为每个历史位置减去最后观测位置；后两维为相邻历史位置差除以 `dt`。第一项速度复制第二项速度。整段历史在预测时已经可见，因此此复制不使用预测区间的信息。

保留最后观测点的时间维度以便广播；不要在此函数中标准化，公共代码会只用训练集统计量完成标准化。

In [ ]:
def make_features(observed, dt):
    """observed: (B,O,2) meters -> features: (B,O,4), using history only."""
    # TODO [features]: Concatenate last-position-centered coordinates and finite-difference velocities; copy velocity at index 1 into index 0.
    raise NotImplementedError("TODO: features")

## B. Elman RNN，20 分

实现 `h_t = tanh(W_x x_t + b_x + W_h h_(t-1))`，以及 `g(h)=W_o h+b_o`。

输入映射用 `Linear(d_x,d_h)`，递归映射用 `Linear(d_h,d_h,bias=False)`，输出映射用 `Linear(d_h,d_y)`。保存 `self.d_h`，递归权重正交初始化，输出权重和偏置置零，使初始预测等于匀速基线。

`f` 返回 `(B,D_h)`，`g` 返回 `(B,2H)`。初始隐状态为 `(B,D_h)` 的零张量，其 device 和 dtype 必须与参数一致。

In [ ]:
class ElmanRNN(nn.Module):
    def __init__(self, d_x, d_h, d_y):
        super().__init__()
        # TODO [elman_init]: Store d_h; create input/recurrent/output Linear layers, orthogonal recurrent weights and a zero-initialized output head.
        raise NotImplementedError("TODO: elman_init")

    def f(self, h, x):
        """h: (B,d_h), x: (B,d_x) -> next hidden state: (B,d_h)."""
        # TODO [elman_f]: Apply tanh to the sum of input and hidden affine projections.
        raise NotImplementedError("TODO: elman_f")

    def g(self, h):
        """Map final hidden state to all future residual coordinates."""
        # TODO [elman_g]: Map hidden states to d_y output coordinates.
        raise NotImplementedError("TODO: elman_g")

    def initial_recurrent_state(self, batch_size):
        # TODO [elman_h0]: Return (batch_size, d_h) zeros on the parameter device and dtype.
        raise NotImplementedError("TODO: elman_h0")

## C. scan，15 分

输入 `xs` 为 `(B,O,D_x)`。沿时间维度 1 逐步调用 `f` 更新隐状态，返回 `(B,O,D_h)`。允许 Python 时间循环；每一步同时处理整个 batch。不能 `detach()`、转 NumPy 或重建破坏计算图的张量。最后隐状态应保留通往早期输入的梯度。

In [ ]:
def scan(f, h, xs):
    """Scan time, retaining autograd: (B,O,d_x) -> (B,O,d_h)."""
    # TODO [scan]: Loop over time dimension 1, update h, and stack states without detaching gradients.
    raise NotImplementedError("TODO: scan")

## D. 多步预测，10 分

公共函数以最后 6 个历史位置（0.5 秒跨度）最小二乘拟合速度 `v_hat`，给出相对末帧的匀速预测 `baseline[k] = v_hat * (k+1) * dt`。

`predict(rnn,xs,baseline,output_scale)` 先初始化隐状态并 `scan`，取最后状态，经 `g` 输出后 reshape 为 `(B,H,2)`。预测公式：

`prediction = baseline + output_scale * residual`

`output_scale=1.0` 表示固定的一米尺度，不是从测试集估计出的参数。返回相对最后观测点的米制位移；绝对位置须再加最后观测位置。所有未来步一次生成，无 teacher forcing，也不把真实未来点回填给模型。

In [ ]:
def predict(rnn, xs, baseline, output_scale):
    """Normalized history + CV forecast -> future relative positions in meters.

    xs: (B,O,4); baseline: (B,H,2); output_scale: a fixed 1-meter scale.
    No future ground-truth values are available to this function.
    """
    # TODO [predict]: Use initial_recurrent_state, scan, and g on the final state; reshape to (B,H,2), scale, and add baseline.
    raise NotImplementedError("TODO: predict")

## E. 单轮训练，15 分

`train_one_epoch` 每次读入 `(xs,baseline,target)`，切换训练模式并将张量移到指定 device。清空梯度，预测，计算 `mean(((prediction-target)/output_scale)^2)`，反向传播，梯度范数裁剪到 1.0，再更新优化器。

返回按样本数加权的 epoch 平均损失（Python float）。最后一个 batch 可能更小，不能直接等权平均各 batch loss。公共代码使用 AdamW，每轮只根据验证 ADE 保存最佳模型；测试集用于最后报告。

In [ ]:
def train_one_epoch(rnn, loader, optimizer, device, output_scale):
    """One epoch; return sample-weighted normalized coordinate MSE."""
    # TODO [train]: Train one epoch: zero_grad, predict, normalized MSE, backward, gradient clipping, optimizer.step; return sample-weighted mean loss.
    raise NotImplementedError("TODO: train")

## F. ADE / FDE，10 分

先计算 `distance[b,k] = sqrt((pred_x-target_x)^2+(pred_y-target_y)^2)`，再返回 `ADE[b]=mean_k(distance[b,k])` 和 `FDE[b]=distance[b,H-1]`。两者形状均为 `(B,)`，单位米。不能用坐标 RMSE 替代 ADE。

In [ ]:
def displacement_errors(prediction, target):
    """Return per-window ADE, FDE: two tensors of shape (B,), in meters."""
    # TODO [metrics]: Compute Euclidean distances; return the time mean and final-step distance for each window.
    raise NotImplementedError("TODO: metrics")

## G. 自定义门控 RNN，15 分

保持相同接口，可按以下 GRU 形式手写门控递归（`*` 为逐元素乘法）：

```text
r = sigmoid(W_xr x + b_r + W_hr h)
z = sigmoid(W_xz x + b_z + W_hz h)
n = tanh(W_xn x + b_n + r * (W_hn h))
h_new = z * h + (1-z) * n
```

输入和递归映射各输出 `3D_h` 维，沿最后一维分为三组。递归映射不加偏置，对三组递归权重分别正交初始化，输出头同样置零。允许提出其他递归结构，但需要说明公式并遵守公共接口。与 Elman 比较时固定数据划分、输入信息和训练预算；不要求门控模型必然优于 Elman。

In [ ]:
class MyRNN(nn.Module):
    """A manually implemented gated RNN (GRU equations)."""

    def __init__(self, d_x, d_h, d_y):
        super().__init__()
        # TODO [custom_init]: Implement your gated or other improved recurrent parameters; use a zero-initialized output head.
        raise NotImplementedError("TODO: custom_init")

    def f(self, h, x):
        # TODO [custom_f]: Implement your recurrent update, for example reset/update gates and a candidate state.
        raise NotImplementedError("TODO: custom_f")

    def g(self, h):
        # TODO [custom_g]: Map hidden states to d_y output coordinates.
        raise NotImplementedError("TODO: custom_g")

    def initial_recurrent_state(self, batch_size):
        # TODO [custom_h0]: Return (batch_size, d_h) zeros on the parameter device and dtype.
        raise NotImplementedError("TODO: custom_h0")

## 4. 实现检查

先完成 A–F 可设 `MODELS="elman"`；完成 G 后设 `MODELS="both"`。
公开检查覆盖物理单位、形状、梯度跨时间传播、独立 batch、训练能否下降及数据隔离。
此处的人工小张量仅用于检查函数正确性；后续训练和测试均使用 PHUMA。

In [ ]:
MODELS = "both"  # "elman", "gated", or "both"
EPOCHS = 100
DEVICE = "cpu"  # Set "cuda" only with a GPU runtime.

In [ ]:
def check(module, model_name):
    torch.manual_seed(9)
    torch.set_num_threads(2)
    times = torch.arange(15) * DT
    history = torch.stack((2 * times, 4 * times), dim=-1)[None].repeat(3, 1, 1)
    features = module.make_features(history, DT)
    assert features.shape == (3, 15, 4)
    assert torch.isfinite(features).all()
    torch.testing.assert_close(module.make_features(history + 20, DT), features, atol=3e-5, rtol=1e-5)
    torch.testing.assert_close(features[:, -1, :2], torch.zeros(3, 2))
    torch.testing.assert_close(features[0, :, 2:], torch.tensor([2.0, 4.0]).expand(15, 2))
    baseline = constant_velocity(history, 15)
    future_times = torch.arange(1, 16) * DT
    exact = torch.stack((2 * future_times, 4 * future_times), dim=-1)
    torch.testing.assert_close(baseline, exact[None].expand(3, 15, 2))
    target = torch.zeros(3, 15, 2)
    shifted = target + torch.tensor([3.0, 4.0])
    ade, fde = module.displacement_errors(shifted, target)
    torch.testing.assert_close(ade, torch.full((3,), 5.0))
    torch.testing.assert_close(fde, torch.full((3,), 5.0))
    shifted.zero_()
    shifted[:, -1, 0] = 15
    ade, fde = module.displacement_errors(shifted, target)
    torch.testing.assert_close(ade, torch.ones(3))
    torch.testing.assert_close(fde, torch.full((3,), 15.0))
    cls = module.ElmanRNN if model_name == "elman" else module.MyRNN
    model = cls(4, 12, 30)
    assert "forward" not in cls.__dict__, "Use f, g, scan as in the assignment."
    forbidden = (nn.RNN, nn.RNNCell, nn.GRU, nn.GRUCell, nn.LSTM, nn.LSTMCell)
    assert not any(isinstance(layer, forbidden) for layer in model.modules())
    xs = torch.randn(3, 15, 4, requires_grad=True)
    hs = module.scan(model.f, model.initial_recurrent_state(3), xs)
    assert hs.shape == (3, 15, 12)
    gradient = torch.autograd.grad(hs[:, -1].square().sum(), xs)[0]
    assert gradient[:, 0].abs().sum() > 0, "scan detached the recurrent history."
    pred = module.predict(model, xs, baseline, 1.0)
    assert pred.shape == (3, 15, 2) and torch.isfinite(pred).all()
    torch.testing.assert_close(pred, baseline)
    double_model = cls(4, 12, 20).double()
    assert double_model.initial_recurrent_state(2).dtype == torch.float64
    default_pred = module.predict(double_model, torch.randn(2, 20, 4, dtype=torch.float64),
                                  torch.zeros(2, 10, 2, dtype=torch.float64), 1.0)
    assert default_pred.shape == (2, 10, 2)
    for i in range(3):
        torch.testing.assert_close(pred[i:i + 1], module.predict(model, xs[i:i + 1], baseline[i:i + 1], 1.0))
    # A tiny learnable residual task catches a missing backward/optimizer step.
    learning_target = baseline + 0.5
    loader = DataLoader(TensorDataset(xs.detach(), baseline, learning_target), batch_size=3)
    before = ((pred.detach() - learning_target) ** 2).mean().item()
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)
    for _ in range(25):
        loss = module.train_one_epoch(model, loader, optimizer, "cpu", 1.0)
        assert isinstance(loss, (float, int)) and loss >= 0
    after = ((module.predict(model, xs, baseline, 1.0) - learning_target) ** 2).mean().item()
    assert after < before * 0.3, f"Training did not fit the tiny task: {before} -> {after}"
    # Unequal batch sizes must still produce a sample-weighted epoch loss.
    unequal_target = baseline + torch.tensor([0.1, 0.2, 2.0])[:, None, None]
    unequal_loader = DataLoader(TensorDataset(xs.detach(), baseline, unequal_target), batch_size=2)
    expected = ((module.predict(model, xs, baseline, 1.0) - unequal_target) ** 2).mean().item()
    actual = module.train_one_epoch(model, unequal_loader, torch.optim.SGD(model.parameters(), lr=0), "cpu", 1.0)
    assert abs(expected - actual) < 1e-5, "Epoch loss must weight by batch size."
    tracks, _ = read_tracks(DATA)
    splits = split_tracks(tracks, 42, 30)
    assert len(set(sum(splits.values(), []))) == sum(map(len, splits.values()))
    for ids in splits.values():
        observed, future, groups = make_windows(tracks, ids, 20, 10, 2)
        assert observed.shape[1:] == (20, 2) and future.shape[1:] == (10, 2)
        assert set(groups) == set(ids)
    # Short chunks cannot be joined to manufacture an otherwise invalid window.
    try:
        make_windows({"source": [torch.zeros(19, 2).numpy(), torch.zeros(19, 2).numpy()]}, ["source"], 20, 10, 2)
        raise AssertionError("Windows crossed a chunk boundary.")
    except ValueError:
        pass
    print(f"PASS: {model_name}: shapes, units, metrics, history gradients, independent batches, training, splits")

from types import SimpleNamespace
implementation = SimpleNamespace(make_features=make_features, ElmanRNN=ElmanRNN,
    MyRNN=MyRNN, scan=scan, predict=predict, train_one_epoch=train_one_epoch,
    displacement_errors=displacement_errors)
assert MODELS in ("both", "elman", "gated")
for model_name in (["elman", "gated"] if MODELS == "both" else [MODELS]):
    check(implementation, model_name)

## 5. 从头训练与最终评估

默认 100 epochs。两个模型都从零输出头开始，相同基线和输入，分别依据验证集选 checkpoint。
测试指标只在训练及模型选择结束后计算。不要根据测试表现反复选择超参数。
每次重跑本单元创建新输出目录，不覆盖旧实验。

In [ ]:
from datetime import datetime
RUN = ROOT / "outputs" / datetime.now().strftime("run_%Y%m%d_%H%M%S_%f")
run_experiment({"elman": ElmanRNN, "gated": MyRNN}, make_features,
    predict, train_one_epoch, displacement_errors,
    argv=["--data", str(DATA), "--output", str(RUN), "--epochs", str(EPOCHS),
          "--model", MODELS, "--device", DEVICE, "--seed", "42"])
report = json.loads((RUN / "metrics.json").read_text())

In [ ]:
table = "| Model | Test ADE (m) | Test FDE (m) | Macro-source ADE (m) |\n"
table += "| --- | ---: | ---: | ---: |\n"
for name, metric in report["test"].items():
    macro = report["test_macro_source"][name]["ADE_m"]
    table += f"| {name} | {metric['ADE_m']:.4f} | {metric['FDE_m']:.4f} | {macro:.4f} |\n"
display(Markdown(table))
for name in report["models"]:
    gain = 100 * (1 - report["test"][name]["ADE_m"] / report["test"]["constant_velocity"]["ADE_m"])
    print(f"{name}: ADE reduction versus CV = {gain:.1f}%")
for filename in ("learning_curves.png", "trajectories.png", "error_by_horizon.png"):
    display(Image(filename=str(RUN / filename)))

## 6. 单条历史的推理与保存

下面只把一个测试窗口的 **20 个历史点** 送给模型，并加回最后观测位置，生成 `(10,2)` 的绝对位置。
模型类型按验证 ADE 选择，不使用测试误差选择。`forecast_history` 也可接收同坐标系、米制、10 Hz 的自有历史。
加载的 checkpoint 为本 notebook 刚训练的文件，包含训练集标准化参数。

In [ ]:
@torch.no_grad()
def forecast_history(history, checkpoint_path):
    saved = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    config = saved["config"]
    history = torch.as_tensor(history, dtype=torch.float32)
    if history.shape != (config["obs_len"], 2) or not torch.isfinite(history).all():
        raise ValueError(f"Expected finite ({config['obs_len']}, 2) history")
    model = {"elman": ElmanRNN, "gated": MyRNN}[saved["model"]](
        4, config["hidden"], 2 * config["pred_len"])
    model.load_state_dict(saved["state_dict"])
    model.eval()
    history = history[None]
    features = (make_features(history, saved["dt"]) - saved["mean"]) / saved["std"]
    baseline = constant_velocity(history, config["pred_len"], saved["dt"])
    relative = predict(model, features, baseline, saved["output_scale"])
    return (relative + history[:, -1:])[0].numpy()

selected_model = min(report["models"], key=lambda n: report["models"][n]["val_ADE_m"])
with np.load(RUN / "test_predictions.npz", allow_pickle=False) as saved_predictions:
    history = saved_predictions["observed"][0].copy()
    expected = saved_predictions[selected_model + "_relative"][0] + history[-1]
forecast = forecast_history(history, RUN / f"{selected_model}_best.pt")
np.testing.assert_allclose(forecast, expected, atol=1e-5, rtol=1e-5)
np.save(RUN / "example_history.npy", history)
np.save(RUN / "example_forecast.npy", forecast)
print("Selected by validation:", selected_model, "forecast shape:", forecast.shape)
print("Future absolute XY positions (meters):\n", forecast)
print("Outputs:", RUN)

## H. 实验与提交，5 分

提交完成的学生 notebook（保留运行输出）及简短分析；若使用脚本则提交 `student.py` 和实验输出。报告至少包含：模型公式、训练配置、验证曲线、基线和两个 RNN 的 ADE/FDE、测试轨迹图、一个失败案例及可能原因。

解释为什么按源动作分组、为什么只用训练集标准化、为什么 RNN 可能优于匀速、为什么更复杂模型未必更好。改变超参数时依据验证集选择；测试集评估后不再反复据此调参。测试图中的失败案例可用于解释，不能用于事后筛除测试数据。

参考教学目标：同一固定划分下 **ADE <= 0.15 米、FDE <= 0.30 米，并优于匀速基线**。依赖版本与硬件允许合理浮动；评分以实现正确、实验规范和分析为主，不把某个小数值作为唯一标准。参考结果来自机器人参考动作，不能据此宣称实机部署精度。

先完成 Elman 后可将 notebook 的 `MODELS` 改为 `"elman"` 进行阶段性实验；全部完成后设回 `"both"`。修改函数定义后要重新运行对应单元，再运行检查与训练。原始学生版的 `NotImplementedError` 提醒你完成相应 TODO。

### 作答区

1. 为什么需要按源动作划分，而不能先滑窗再随机划分？
2. 训练集归一化与仅用历史构造速度分别避免什么信息泄漏？
3. 对比两个 RNN 与匀速基线的 ADE / FDE，解释差异。
4. 选一个误差较大的测试案例，讨论停止、转向或动作意图对预测的影响。
5. 说明数据来源与本实验结果不能直接迁移到哪些实际任务。

请在此填写分析。